# Task 4 — Parte A: otimização de hiperparâmetros

Este notebook faz a **primeira metade da Task 4**: escolher os modelos mais promissores da Task 3, otimizar os seus hiperparâmetros e perceber se a otimização traz uma melhoria **real e estável**. No fim, guarda o que a Parte B (limiar, teste e interpretação) precisa.

**Como funciona**

- Usa o `avaliar_folds` e a restante infraestrutura da Task 2 (`avaliacao.py`) e o pré-processamento da Task 3 (`ablacao.py`): 4 folds temporais, recall com 200 alertas por 1000 sessões como métrica principal. O pré-processamento e o modelo são **reajustados em cada fold, só com o treino desse fold**. O teste (out–dez/2021) é removido ao carregar os dados e nunca é usado aqui.
- As funções de pesquisa e de escolha estão em `otimizacao.py`. O notebook só chama essas funções e mostra os resultados.
- **Os resultados vão sendo guardados** em `Resultados/` durante a pesquisa. Se o computador adormecer ou o kernel falhar, volta-se a correr a mesma célula e só se calcula o que falta.

| Secção | O que responde |
|---|---|
| 0 | Configuração e dados |
| 1 | Que modelos otimizar, e porquê |
| 2 | Procedimento de pesquisa e espaços de pesquisa |
| 3 | As três pesquisas (logística, boosting, Random Forest) |
| 4 | Inicial contra otimizado: média, variabilidade e evolução no tempo |
| 5 | Verificação sem otimismo: escolher com F1–F3 e avaliar em F4 |
| 6 | Qual é o modelo probabilístico mais forte |
| 7 | Ficheiros para a Parte B |
| 8 | Conclusões: o que escrever no relatório |

## 0. Configuração e dados

In [7]:
import json
import warnings
from pathlib import Path
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"   # põe o número de núcleos do teu computador
import numpy as np
import pandas as pd

import ablacao as ab
import otimizacao as ot
from avaliacao import avaliar_folds, comparar

warnings.filterwarnings("ignore", category=FutureWarning)   # os avisos de convergência continuam visíveis
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

DATA_PATH = Path("Charging_Data_educational.csv")
PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

X_ALERTAS = 200      # orçamento de alertas por 1000 sessões (secção 8.4 do relatório)
SEED = 0             # semente da pesquisa aleatória
AMOSTRA = 1          # 1 = todos os dados. Para um ensaio rápido, 8 (volta a 1 para os resultados finais)
ESPACO = None        # sem informação espacial: com o histórico, o posto não acrescenta nada (secção 9.1 e 9.2)
GRUPOS_TODOS = ab.CONFIGURACOES["4 + histórico do utilizador"]
MODELOS = ["logistica", "boosting", "random_forest"]
NOMES = {"logistica": "Regressão logística", "boosting": "Boosting", "random_forest": "Random Forest"}

In [8]:
df = ab.carregar_dev(DATA_PATH)            # só treino e validação (dez/2019 – set/2021)
assert df["Start Time"].max() < pd.Timestamp("2021-10-01"), "o teste não pode estar carregado"
if AMOSTRA > 1:
    df = df.iloc[::AMOSTRA].reset_index(drop=True)
    print(f"ATENÇÃO: ensaio rápido com 1 em cada {AMOSTRA} sessões. Não usar estes resultados.")
print(f"{len(df):,} sessões | {df['Start Time'].min():%Y-%m-%d} a {df['Start Time'].max():%Y-%m-%d} | prevalência {df['is_Abnormal'].mean():.1%}")


def metricas_fold(res):
    return pd.Series({"recall@200 (média)": res["recall"].mean(), "desvio": res["recall"].std(ddof=1),
                      "pior fold": res["recall"].min(), "PR-AUC norm. (média)": res["pr_auc_norm"].mean(),
                      "ROC-AUC (média)": res["roc_auc"].mean()})

360,996 sessões | 2019-12-31 a 2021-09-30 | prevalência 18.8%


## 1. Que modelos otimizar, e porquê

O enunciado pede **dois ou três** modelos promissores. A escolha apoia-se nos resultados da Task 3 (combinação 4, recall com 200 alertas por 1000 sessões, média dos 4 folds):

| Modelo (Task 3) | Recall | Pior fold | PR-AUC norm. | Tempo (4 folds) | Decisão |
|---|---|---|---|---|---|
| Random Forest | 0,588 | 0,576 | 0,547 | 255 s | **Otimizar.** Melhor recall e melhor PR-AUC; é o modelo preferido pela regra da secção 8.9. |
| Boosting | 0,586 | 0,572 | 0,544 | 23 s | **Otimizar.** Quase empata com a Random Forest, custa dez vezes menos e tem mais hiperparâmetros com efeito (profundidade, folhas, regularização), logo mais margem para melhorar. |
| Regressão logística | 0,580 | 0,566 | 0,521 | 21 s | **Otimizar.** É o melhor modelo interpretável e a referência probabilística: mostra se os modelos complexos justificam o custo. Já se viu na Task 3 que o `C` não altera o recall, por isso é um teste da hipótese "o modelo está limitado pela informação, não pela configuração". |
| Árvore de decisão | 0,578 | 0,564 | 0,525 | 24 s | Não. É dominada pela Random Forest, que é um conjunto destas árvores. |
| Naive Bayes | 0,574 | 0,559 | 0,498 | 2 s | Não. É o mais fraco e as probabilidades são demasiado extremas (a independência não se cumpre). |
| Votação (logística + árvore) | 0,583 | 0,572 | 0,536 | 45 s | Não. Combina modelos já afinados: otimizá-la seria otimizar os seus componentes. |

**Expectativa.** Na Task 3 as diferenças entre modelos foram inferiores a 0,015 e a afinação simples trouxe, no máximo, +0,003. É provável que a otimização acrescente pouco. Concluir que **não trouxe uma melhoria real** é um resultado válido, desde que esteja demonstrado.

## 2. Procedimento de pesquisa

| Elemento | Decisão |
|---|---|
| Dados | Só treino e validação (dez/2019 a set/2021). O teste não é carregado. |
| Ordem temporal | Os mesmos 4 folds de janela crescente da Task 2 (validações trimestrais de out/2020 a set/2021). |
| Pré-processamento | Reajustado em cada fold com o treino desse fold (`fabricar_scorer`). Nada é calculado com a validação. |
| Variáveis | Combinação 4, sem informação espacial (como a Random Forest e o boosting na Task 3). |
| Método | **Pesquisa aleatória** com semente fixa (`SEED`). Há cinco a sete hiperparâmetros por modelo, e a pesquisa aleatória cobre melhor o espaço do que uma grelha com o mesmo número de corridas. Na logística, com dois hiperparâmetros, usa-se a **grelha** completa. |
| Métrica a otimizar | **Recall com 200 alertas por 1000 sessões**, a métrica principal do projeto (secção 8.4), com a regra da secção 8.9: diferenças inferiores a 0,005 são empate, **desempatado pela PR-AUC normalizada**. Por **parcimónia**, só se abandona a configuração inicial se o desempate a superar em mais de 0,005 de PR-AUC normalizada; senão a diferença é ruído e mantém-se a inicial. |
| Comparação | Cada configuração é comparada com a **inicial** (a que saiu da Task 3), fold a fold, com a regra da secção 8.9: maior média, melhor em pelo menos 3 de 4 folds e pior fold não inferior. |
| Poupar tempo | Na Random Forest usam-se 100 árvores durante a pesquisa e 200 na avaliação final. A configuração inicial é avaliada nas mesmas condições, por isso a comparação é justa. |

**Porque o recall e não a PR-AUC.** O recall com 200 alertas é o critério de decisão do projeto, mas num fold varia por acaso cerca de 0,004 a 0,006 (secção 8.9). Por isso diferenças inferiores a 0,005 contam como empate e a PR-AUC normalizada, que usa todas as previsões, desempata.

**Cuidados**

- Os mesmos folds servem para **escolher** e para **comparar**, por isso o ganho medido é **ligeiramente otimista**. A secção 5 faz uma verificação sem esse otimismo: escolhe-se só com F1–F3 e avalia-se em F4.
- Os resultados ficam em `Resultados/pesquisa_<modelo>_amostra<N>.csv`. Apaga o ficheiro se alterares o espaço de pesquisa.

In [9]:
# Configuração de partida (a que saiu da Task 3; o que não é indicado vem de ab.HIPERPARAMETROS)
INICIAL = {
    "logistica":     {"C": 0.1, "class_weight": None},
    "boosting":      {"learning_rate": 0.05, "max_iter": 200},
    "random_forest": {"min_samples_leaf": 50, "max_features": 0.5},
}

# Espaços de pesquisa
ESPACOS = {
    "logistica": {"C": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10],
                  "class_weight": [None, "balanced"]},
    "boosting": {"learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
                 "max_iter": [100, 150, 200, 300, 400],
                 "max_leaf_nodes": [7, 15, 31, 63],
                 "max_depth": [None, 3, 5, 8],
                 "min_samples_leaf": [20, 50, 100, 200, 500],
                 "l2_regularization": [0, 0.1, 1, 10],
                 "max_features": [0.5, 0.8, 1.0]},
    "random_forest": {"min_samples_leaf": [10, 25, 50, 100],
                      "max_features": ["sqrt", 0.3, 0.5, 0.7],
                      "max_depth": [None, 12, 20],
                      "max_samples": [0.3, 0.5, 0.7, None]},
}

# Número de configurações por modelo (a grelha da logística tem 16 pontos)
N_CONFIG = {"logistica": 16, "boosting": 40, "random_forest": 16}
# Só durante a pesquisa: menos árvores na Random Forest, para poupar tempo
FIXOS_PESQUISA = {"random_forest": {"n_estimators": 100}}

pd.DataFrame({NOMES[m]: {"configurações testadas": N_CONFIG[m],
                         "pontos no espaço": int(np.prod([len(v) for v in ESPACOS[m].values()])),
                         "hiperparâmetros": ", ".join(ESPACOS[m])} for m in MODELOS}).T

,configurações testadas,pontos no espaço,hiperparâmetros
Regressão logística,16,16,"C, class_weight"
Boosting,40,24000,"learning_rate, max_iter, max_leaf_nodes, max_depth, min_samples_leaf, l2_regularizatio..."
Random Forest,16,192,"min_samples_leaf, max_features, max_depth, max_samples"


**Porque estes intervalos**

- **Logística.** `C` (inverso da regularização) de 0,003 a 10, em escala logarítmica, que cobre desde muito regularizada até quase sem regularização. `class_weight` testa se dar mais peso às falhas ajuda; na Task 3 não ajudou com o `C` fixo.
- **Boosting.** Taxa de aprendizagem e número de iterações (andam juntas: taxa baixa pede mais iterações); `max_leaf_nodes` e `max_depth` controlam a complexidade de cada árvore; `min_samples_leaf` e `l2_regularization` limitam o sobreajuste; `max_features` sorteia uma fração das variáveis em cada divisão. O valor inicial (taxa 0,05, 200 iterações, 31 folhas, folha mínima 100) está dentro do espaço.
- **Random Forest.** `min_samples_leaf` e `max_depth` controlam o tamanho das árvores; `max_features` e `max_samples` (tamanho da amostra bootstrap) controlam a diversidade entre árvores, que é o que faz a floresta melhorar uma árvore isolada.

## 3. As três pesquisas

In [10]:
def mostrar(tabela, n=10):
    """Tabela das melhores configurações (a inicial vem marcada com *)."""
    t = tabela.sort_values("recall_media", ascending=False).head(n).copy()
    t["configuração"] = np.where(t["inicial"], "* ", "") + t["hp"]
    return t[["configuração", "recall_media", "recall_desvio", "recall_pior", "prauc_media", "roc_media", "tempo_s"]] \
        .rename(columns={"recall_media": "recall@200 (média)", "recall_desvio": "desvio", "recall_pior": "pior fold",
                         "prauc_media": "PR-AUC norm.", "roc_media": "ROC-AUC", "tempo_s": "tempo (s)"}).round(4)

tabelas = {}


def correr_pesquisa(m):
    lista = ot.candidatos(ESPACOS[m], N_CONFIG[m], seed=SEED)
    tabelas[m] = ot.pesquisar(df, m, lista, GRUPOS_TODOS, ESPACO, X_ALERTAS, fixos=FIXOS_PESQUISA.get(m),
                              inicial=INICIAL[m], ficheiro=PASTA / f"pesquisa_{m}_amostra{AMOSTRA}.csv")
    return mostrar(tabelas[m])

### 3.1 Regressão logística (grelha)

In [11]:
correr_pesquisa("logistica")

,configuração,recall@200 (média),desvio,pior fold,PR-AUC norm.,ROC-AUC,tempo (s)
1,"{""C"": 0.003, ""class_weight"": null}",0.5793,0.0142,0.5657,0.5190,0.8297,6.4388
3,"{""C"": 0.01, ""class_weight"": null}",0.5791,0.0138,0.5669,0.5198,0.8298,7.6765
5,"{""C"": 0.03, ""class_weight"": null}",0.5790,0.0139,0.5665,0.5200,0.8298,7.9790
10,"{""C"": 1, ""class_weight"": null}",0.5790,0.0136,0.5663,0.5202,0.8297,6.1211
8,"{""C"": 0.3, ""class_weight"": null}",0.5790,0.0136,0.5663,0.5202,0.8297,6.0263
12,"{""C"": 3, ""class_weight"": null}",0.5789,0.0136,0.5663,0.5202,0.8297,5.8975
14,"{""C"": 10, ""class_weight"": null}",0.5789,0.0136,0.5663,0.5202,0.8297,5.8716
0,"* {""C"": 0.1, ""class_weight"": null}",0.5788,0.0136,0.5664,0.5202,0.8298,6.1865
6,"{""C"": 0.03, ""class_weight"": ""balanced""}",0.5784,0.0144,0.5643,0.5165,0.8298,6.2661
9,"{""C"": 0.3, ""class_weight"": ""balanced""}",0.5784,0.0144,0.5645,0.5166,0.8298,6.7471


**Conclusão**. As 16 configurações ficam entre 0,578 e 0,579 de recall, uma diferença inferior a 0,005, por isso são todas empate. A configuração inicial (C = 0,1) mantém-se. class_weight="balanced" é ligeiramente pior.

### 3.2 Boosting (pesquisa aleatória, 40 configurações)

In [12]:
correr_pesquisa("boosting")

[boosting] 1/41  recall 0.5862  PR-AUC norm. 0.544  (19 s)  {"learning_rate": 0.05, "max_iter": 200}
[boosting] 2/41  recall 0.5850  PR-AUC norm. 0.539  (11 s)  {"l2_regularization": 0, "learning_rate": 0.05, "max_depth": 3, "max_features": 0.5, "max_iter": 150, "max_leaf_nodes": 31, "min_samples_leaf": 100}
[boosting] 3/41  recall 0.5834  PR-AUC norm. 0.540  (29 s)  {"l2_regularization": 0.1, "learning_rate": 0.08, "max_depth": 8, "max_features": 1.0, "max_iter": 400, "max_leaf_nodes": 63, "min_samples_leaf": 500}
[boosting] 4/41  recall 0.5845  PR-AUC norm. 0.542  (11 s)  {"l2_regularization": 0.1, "learning_rate": 0.08, "max_depth": null, "max_features": 1.0, "max_iter": 200, "max_leaf_nodes": 15, "min_samples_leaf": 20}
[boosting] 5/41  recall 0.5842  PR-AUC norm. 0.538  (11 s)  {"l2_regularization": 10, "learning_rate": 0.03, "max_depth": 3, "max_features": 0.8, "max_iter": 200, "max_leaf_nodes": 15, "min_samples_leaf": 200}
[boosting] 6/41  recall 0.5827  PR-AUC norm. 0.535  (9 s

,configuração,recall@200 (média),desvio,pior fold,PR-AUC norm.,ROC-AUC,tempo (s)
25,"{""l2_regularization"": 0.1, ""learning_rate"": 0.02, ""max_depth"": null, ""max_features"": 0...",0.5882,0.0132,0.5750,0.5469,0.8359,28.6916
10,"{""l2_regularization"": 1, ""learning_rate"": 0.03, ""max_depth"": 8, ""max_features"": 0.5, ""...",0.5877,0.0136,0.5746,0.5461,0.8357,15.7102
37,"{""l2_regularization"": 0, ""learning_rate"": 0.02, ""max_depth"": 5, ""max_features"": 0.8, ""...",0.5868,0.0135,0.5732,0.5445,0.8355,44.6830
29,"{""l2_regularization"": 0.1, ""learning_rate"": 0.03, ""max_depth"": 5, ""max_features"": 0.5,...",0.5867,0.0137,0.5734,0.5445,0.8354,22.4784
31,"{""l2_regularization"": 0, ""learning_rate"": 0.03, ""max_depth"": 5, ""max_features"": 0.5, ""...",0.5867,0.0134,0.5735,0.5444,0.8355,24.9761
9,"{""l2_regularization"": 10, ""learning_rate"": 0.08, ""max_depth"": 5, ""max_features"": 0.5, ...",0.5867,0.0131,0.5742,0.5445,0.8353,13.9899
15,"{""l2_regularization"": 10, ""learning_rate"": 0.03, ""max_depth"": 5, ""max_features"": 0.5, ...",0.5865,0.0130,0.5742,0.5445,0.8354,21.6418
13,"{""l2_regularization"": 1, ""learning_rate"": 0.08, ""max_depth"": null, ""max_features"": 1.0...",0.5864,0.0137,0.5734,0.5437,0.8349,14.2920
0,"* {""learning_rate"": 0.05, ""max_iter"": 200}",0.5862,0.0141,0.5730,0.5443,0.8349,18.8015
28,"{""l2_regularization"": 0, ""learning_rate"": 0.1, ""max_depth"": null, ""max_features"": 0.5,...",0.5859,0.0130,0.5745,0.5416,0.8339,25.9638


**Conclusão**. A melhor configuração tem recall 0,5882, contra 0,5862 da inicial (+0,002, dentro do ruído). A PR-AUC normalizada sobe 0,003, abaixo de 0,005. Mantém-se a configuração inicial.

### 3.3 Random Forest (pesquisa aleatória, 16 configurações)

In [13]:
correr_pesquisa("random_forest")

[random_forest] 1/17  recall 0.5878  PR-AUC norm. 0.546  (205 s)  {"max_features": 0.5, "min_samples_leaf": 50}
[random_forest] 2/17  recall 0.5870  PR-AUC norm. 0.547  (151 s)  {"max_depth": 12, "max_features": 0.5, "max_samples": null, "min_samples_leaf": 50}
[random_forest] 3/17  recall 0.5880  PR-AUC norm. 0.544  (51 s)  {"max_depth": 12, "max_features": "sqrt", "max_samples": 0.7, "min_samples_leaf": 50}
[random_forest] 4/17  recall 0.5870  PR-AUC norm. 0.545  (71 s)  {"max_depth": 20, "max_features": 0.5, "max_samples": 0.3, "min_samples_leaf": 100}
[random_forest] 5/17  recall 0.5868  PR-AUC norm. 0.547  (61 s)  {"max_depth": 12, "max_features": 0.5, "max_samples": 0.3, "min_samples_leaf": 25}
[random_forest] 6/17  recall 0.5871  PR-AUC norm. 0.547  (202 s)  {"max_depth": 12, "max_features": 0.7, "max_samples": null, "min_samples_leaf": 50}
[random_forest] 7/17  recall 0.5868  PR-AUC norm. 0.541  (35 s)  {"max_depth": 12, "max_features": "sqrt", "max_samples": 0.5, "min_samples_

,configuração,recall@200 (média),desvio,pior fold,PR-AUC norm.,ROC-AUC,tempo (s)
7,"{""max_depth"": null, ""max_features"": 0.3, ""max_samples"": 0.3, ""min_samples_leaf"": 50}",0.5882,0.0134,0.5755,0.5452,0.8347,41.9620
9,"{""max_depth"": 20, ""max_features"": 0.3, ""max_samples"": 0.3, ""min_samples_leaf"": 25}",0.5881,0.0129,0.5769,0.5451,0.8341,46.0543
2,"{""max_depth"": 12, ""max_features"": ""sqrt"", ""max_samples"": 0.7, ""min_samples_leaf"": 50}",0.5880,0.0138,0.5756,0.5443,0.8350,51.3054
8,"{""max_depth"": 20, ""max_features"": 0.3, ""max_samples"": null, ""min_samples_leaf"": 25}",0.5879,0.0122,0.5767,0.5460,0.8338,112.3071
13,"{""max_depth"": 20, ""max_features"": 0.3, ""max_samples"": null, ""min_samples_leaf"": 50}",0.5878,0.0129,0.5764,0.5466,0.8347,108.6379
0,"* {""max_features"": 0.5, ""min_samples_leaf"": 50}",0.5878,0.0133,0.5762,0.5462,0.8341,204.5700
12,"{""max_depth"": 20, ""max_features"": ""sqrt"", ""max_samples"": 0.7, ""min_samples_leaf"": 100}",0.5877,0.0130,0.5757,0.5422,0.8347,48.1032
14,"{""max_depth"": 20, ""max_features"": 0.5, ""max_samples"": 0.7, ""min_samples_leaf"": 50}",0.5875,0.0129,0.5759,0.5465,0.8344,144.5230
5,"{""max_depth"": 12, ""max_features"": 0.7, ""max_samples"": null, ""min_samples_leaf"": 50}",0.5871,0.0141,0.5746,0.5470,0.8351,201.9652
1,"{""max_depth"": 12, ""max_features"": 0.5, ""max_samples"": null, ""min_samples_leaf"": 50}",0.5870,0.0136,0.5739,0.5474,0.8354,150.8350


**Conclusão**. As dez melhores configurações ficam entre 0,587 e 0,588, e a inicial tem 0,5878. Mantém-se a inicial. Há configurações com o mesmo recall em cerca de 42 s, contra cerca de 205 s da inicial, mas isso não é melhoria de desempenho.

## 4. Inicial contra otimizado

Para cada modelo, a configuração escolhida pela regra (a de maior PR-AUC entre as que ficam a menos de 0,005 do melhor recall, com a parcimónia da secção 2) é **avaliada de novo** com as condições finais (Random Forest com 200 árvores), tal como a inicial. A comparação usa a regra da secção 8.9. Se a escolhida for a própria inicial, mantém-se a inicial.

In [14]:
escolhida, final = {}, {}      # final[m] = {"inicial": (res, scores), "otimizado": (res, scores)}
for m in MODELOS:
    linha = ot.escolher(tabelas[m])
    escolhida[m] = json.loads(linha["hp"])
    e_a_inicial = bool(linha["inicial"])
    ini = INICIAL[m]
    final[m] = {"inicial": avaliar_folds(df, ab.fabricar_scorer(m, GRUPOS_TODOS, ESPACO, **ini), X_ALERTAS, devolver_scores=True)}
    final[m]["otimizado"] = final[m]["inicial"] if e_a_inicial else \
        avaliar_folds(df, ab.fabricar_scorer(m, GRUPOS_TODOS, ESPACO, **escolhida[m]), X_ALERTAS, devolver_scores=True)
    print(f"{NOMES[m]:<20} inicial {ini}\n{'':<20} escolhida {escolhida[m]}" + ("  (é a inicial)" if e_a_inicial else ""))

Regressão logística  inicial {'C': 0.1, 'class_weight': None}
                     escolhida {'C': 0.1, 'class_weight': None}  (é a inicial)
Boosting             inicial {'learning_rate': 0.05, 'max_iter': 200}
                     escolhida {'learning_rate': 0.05, 'max_iter': 200}  (é a inicial)
Random Forest        inicial {'min_samples_leaf': 50, 'max_features': 0.5}
                     escolhida {'max_features': 0.5, 'min_samples_leaf': 50}  (é a inicial)


In [15]:
linhas = {}
for m in MODELOS:
    linhas[(NOMES[m], "inicial")] = metricas_fold(final[m]["inicial"][0])
    linhas[(NOMES[m], "otimizado")] = metricas_fold(final[m]["otimizado"][0])
pd.DataFrame(linhas).T.round(4)

recall@200 (média)  desvio  pior fold  PR-AUC norm. (média)  ROC-AUC (média)
Regressão logística inicial                0.5788  0.0136     0.5664                0.5202           0.8298
                    otimizado              0.5788  0.0136     0.5664                0.5202           0.8298
Boosting            inicial                0.5862  0.0141     0.5730                0.5443           0.8349
                    otimizado              0.5862  0.0141     0.5730                0.5443           0.8349
Random Forest       inicial                0.5877  0.0138     0.5759                0.5467           0.8344
                    otimizado              0.5877  0.0138     0.5759                0.5467           0.8344

In [16]:
# Regra da secção 8.9: o otimizado (A) contra o inicial (B), fold a fold
pd.DataFrame({NOMES[m]: ot.veredicto(final[m]["otimizado"][0], final[m]["inicial"][0]) for m in MODELOS}).T.round(4)

,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B,melhoria real e estável (regra 8.9)
Regressão logística,0.0,0.0,0,0.566434,0.566434,False
Boosting,0.0,0.0,0,0.57299,0.57299,False
Random Forest,0.0,0.0,0,0.575858,0.575858,False


In [17]:
# Evolução temporal: recall por fold, inicial e otimizado, e a diferença
por_fold = {}
for m in MODELOS:
    i, o = final[m]["inicial"][0]["recall"], final[m]["otimizado"][0]["recall"]
    por_fold[(NOMES[m], "inicial")] = i
    por_fold[(NOMES[m], "otimizado")] = o
    por_fold[(NOMES[m], "diferença")] = o - i
pd.DataFrame(por_fold).T.round(4)

F1      F2      F3      F4
Regressão logística inicial    0.5886  0.5678  0.5924  0.5664
                    otimizado  0.5886  0.5678  0.5924  0.5664
                    diferença  0.0000  0.0000  0.0000  0.0000
Boosting            inicial    0.5974  0.5751  0.5993  0.5730
                    otimizado  0.5974  0.5751  0.5993  0.5730
                    diferença  0.0000  0.0000  0.0000  0.0000
Random Forest       inicial    0.5974  0.5759  0.6016  0.5759
                    otimizado  0.5974  0.5759  0.6016  0.5759
                    diferença  0.0000  0.0000  0.0000  0.0000

In [18]:
# O que a pesquisa encontrou de melhor, sem a regra de parcimónia, contra a inicial (mesmas condições de pesquisa)
linhas = {}
for m in MODELOS:
    t = tabelas[m]
    ini = t[t["inicial"].astype(bool)].iloc[0]
    melhor = t.sort_values("recall_media", ascending=False).iloc[0]
    fs = ot.folds_da_tabela(t)
    d = np.array([melhor[f"recall_{f}"] - ini[f"recall_{f}"] for f in fs])
    linhas[NOMES[m]] = {"recall inicial": ini["recall_media"],
                        "recall melhor da pesquisa": melhor["recall_media"],
                        "diferença média": d.mean(),
                        **{f"dif. {f}": x for f, x in zip(fs, d)},
                        "folds em que melhora": int((d > 0).sum()),
                        "melhor é a inicial": bool(melhor["inicial"])}
pd.DataFrame(linhas).T.round(4)

,recall inicial,recall melhor da pesquisa,diferença média,dif. F1,dif. F2,dif. F3,dif. F4,folds em que melhora,melhor é a inicial
Regressão logística,0.578823,0.579312,0.000488,0.001082,0.000605,0.000954,-0.000687,3,False
Boosting,0.586207,0.588213,0.002006,-0.000098,0.003933,0.002193,0.001998,3,False
Random Forest,0.587772,0.588159,0.000387,-0.000197,0.001664,0.000954,-0.000874,2,False


**Conclusão**. Nos três modelos a configuração escolhida é a inicial, por isso as diferenças são nulas em todos os folds e a regra 8.9 não encontra melhoria. O melhor da pesquisa ganha entre +0,0004 e +0,0020 de recall, sempre abaixo de 0,005 e sem repetir-se em todos os folds

## 5. Verificação sem otimismo: escolher com F1–F3 e avaliar em F4

Os mesmos folds servem para escolher e comparar, o que favorece o modelo escolhido. Para medir isso, repete-se a escolha usando **só os folds F1–F3** (sem olhar para F4) e vê-se quanto vale essa configuração em F4, contra a inicial em F4. Não custa tempo, porque usa as tabelas já calculadas na secção 3 (mesmas condições de pesquisa).

In [19]:
linhas = {}
for m in MODELOS:
    t = tabelas[m]
    e3 = ot.escolher(t, folds=["F1", "F2", "F3"])
    ini = t[t["inicial"]].iloc[0]
    linhas[NOMES[m]] = {"escolhida só com F1–F3": e3["hp"],
                        "recall F4 (escolhida)": e3["recall_F4"],
                        "recall F4 (inicial)": ini["recall_F4"],
                        "diferença em F4": e3["recall_F4"] - ini["recall_F4"],
                        "é a inicial": bool(e3["inicial"])}
pd.DataFrame(linhas).T

,escolhida só com F1–F3,recall F4 (escolhida),recall F4 (inicial),diferença em F4,é a inicial
Regressão logística,"{""C"": 0.1, ""class_weight"": null}",0.566434,0.566434,0.0,True
Boosting,"{""learning_rate"": 0.05, ""max_iter"": 200}",0.57299,0.57299,0.0,True
Random Forest,"{""max_features"": 0.5, ""min_samples_leaf"": 50}",0.576361,0.576361,0.0,True


**Conclusão.** Escolhendo só com F1–F3, a regra mantém a configuração inicial nos três modelos, e a diferença em F4 é 0. A conclusão não depende de a escolha ter sido feita nos mesmos folds em que se compara.

## 6. Qual é o modelo probabilístico mais forte

A Parte B vai escolher um limiar sobre as **probabilidades** previstas, por isso interessa o modelo com melhor ordenação e probabilidades fiáveis. Para cada modelo otimizado (já com os resultados da secção 4) calculam-se:

- recall@200, PR-AUC normalizada e ROC-AUC: **qualidade da ordenação**;
- **Brier score** e **log-loss**: qualidade das probabilidades (menor é melhor);
- **ECE**: erro de calibração esperado, a diferença média entre a probabilidade prevista e a frequência observada de falhas (menor é melhor).

In [20]:
linhas = {}
for m in MODELOS:
    res, sc = final[m]["otimizado"]
    linhas[NOMES[m]] = {**metricas_fold(res).to_dict(), **ot.metricas_probabilisticas(sc)}
tabela_prob = pd.DataFrame(linhas).T
tabela_prob.round(4)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média),Brier,log-loss,ECE
Regressão logística,0.5788,0.0136,0.5664,0.5202,0.8298,0.1044,0.3453,0.0138
Boosting,0.5862,0.0141,0.5730,0.5443,0.8349,0.1018,0.3370,0.0060
Random Forest,0.5877,0.0138,0.5759,0.5467,0.8344,0.1016,0.3368,0.0057


In [21]:
# Critério (o mesmo da secção 8.9): maior recall médio; empate (< 0,005) desempatado pela PR-AUC normalizada.
melhor = tabela_prob["recall@200 (média)"].max()
empate = tabela_prob[tabela_prob["recall@200 (média)"] >= melhor - ot.TOLERANCIA]
MODELO_PROB = empate["PR-AUC norm. (média)"].astype(float).idxmax()
chave_prob = [m for m in MODELOS if NOMES[m] == MODELO_PROB][0]
print("Modelo probabilístico mais forte pelo critério:", MODELO_PROB)
print("Candidatos empatados:", list(empate.index))
# Se preferires outro por razões de calibração ou interpretabilidade, indica-o aqui e justifica no relatório:
# chave_prob = "logistica"

Modelo probabilístico mais forte pelo critério: Random Forest
Candidatos empatados: ['Boosting', 'Random Forest']


**Conclusão.** A Random Forest é o modelo mais forte (recall 0,5877, PR-AUC normalizada 0,5467, Brier 0,1016, ECE 0,0057), em empate prático com o boosting. Os três têm erro de calibração baixo (ECE entre 0,006 e 0,014). Os modelos de árvores não ficam menos calibrados do que a logística.

## 7. Ficheiros para a Parte B

A Parte B precisa de duas coisas: as **previsões de validação** de cada fold (cada fold previsto por um modelo ajustado só com dados anteriores, por isso o limiar pode ser escolhido nelas sem ver o teste) e os **hiperparâmetros** a usar no modelo final. É a Parte B que treina o modelo final com todo o período de desenvolvimento.

In [22]:
scores_otimizados = {m: final[m]["otimizado"][1] for m in MODELOS}
ot.guardar_para_B(PASTA, df, escolhida, scores_otimizados, chave_prob,
                  extra={"hiperparametros_iniciais": INICIAL, "alertas_por_1000": X_ALERTAS, "amostra": AMOSTRA})
for f in sorted(PASTA.glob("task4_*")):
    print(f"{f.name:<45} {f.stat().st_size / 1e6:6.2f} MB")
previsoes = pd.read_csv(PASTA / f"task4_oof_validacao_{chave_prob}.csv.gz")
print(previsoes.groupby("fold").size().to_dict(), "| colunas:", list(previsoes.columns))
previsoes.head(3)

task4_modelos_escolhidos.json                   0.00 MB
task4_oof_validacao_boosting.csv.gz             3.85 MB
task4_oof_validacao_logistica.csv.gz            3.84 MB
task4_oof_validacao_random_forest.csv.gz        3.86 MB
{'F1': 60364, 'F2': 41797, 'F3': 57542, 'F4': 75769} | colunas: ['fold', 'linha_df', 'Start Time', 'y', 'score']


,fold,linha_df,Start Time,y,score
0,F1,125524,2020-10-01 00:00:02,0,0.021859
1,F1,125525,2020-10-01 00:00:48,0,0.193312
2,F1,125526,2020-10-01 00:03:13,0,0.216153


**Conclusão.** Ficam guardados os hiperparâmetros escolhidos e as previsões de validação dos três modelos, para a escolha do limiar na Parte B.